# Master Thesis Analysis: LLM Knowledge Cutoffs and Python Library Diffusion

**Author:** Roland Tuboly  
**Project:** Empirical analysis of whether LLM training cutoffs (e.g., September 2021) create an adoption advantage for libraries released just before the cutoff.

## 1. Research Context and Objective
This notebook implements a **Regression Discontinuity Design (RDD)** to study the diffusion of Python libraries around documented LLM knowledge cutoffs. The central hypothesis is that libraries released shortly before a cutoff are better represented in the model's pretrained knowledge, leading to wider diffusion (PyPI downloads and GitHub usage) compared to libraries released shortly after.

### Core Components:
- **Running Variable:** Weeks from the library's initial release to the LLM training cutoff.
- **Cutoff:** September 27, 2021 (Primary).
- **Treatment:** Release date falling *before* the cutoff.
- **Outcomes:** Cumulative PyPI downloads and GitHub imports at 12, 26, and 52-week horizons.
- **Methodology:** Local linear RDD with a triangular kernel and donut-hole specifications.

## 2. Setup and Configuration
We first import necessary libraries and define shared parameters used throughout the analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path
from scipy.stats import norm
import rdrobust
from IPython.display import Image, display

# Paths
BASE_DIR = Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERM_DIR = BASE_DIR / "data" / "intermediate"
FINAL_DIR = BASE_DIR / "data" / "final"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in [INTERM_DIR, FINAL_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Analysis Parameters
WINDOW_WEEKS = 52
DONUT_WEEKS = [-2, -1, 0, 1, 2]
HORIZON_WEEKS = 52
MAIN_CUTOFF_NAME = "Main_2021"
MAIN_CUTOFF_DATE = pd.Timestamp("2021-09-27")

# Cutoff Environments for placebo/multi-cutoff tests
CUTOFFS = {
    "Placebo_2018": pd.Timestamp("2018-09-24"),
    "Placebo_2019": pd.Timestamp("2019-09-30"),
    "Placebo_2020": pd.Timestamp("2020-09-28"),
    MAIN_CUTOFF_NAME: MAIN_CUTOFF_DATE,
    "Adoption_2023": pd.Timestamp("2023-01-02")
}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### Utility Functions
Consistent variable definitions and estimation wrappers are crucial for reproducibility.

In [ ]:
def normalize_name(s: pd.Series) -> pd.Series:
    """Normalize package/library names for consistent merging."""
    return (
        s.astype(str)
        .str.lower()
        .str.replace("_", "-", regex=False)
        .str.strip()
    )

def check_monday_alignment(df: pd.DataFrame, date_col: str = "week_start") -> None:
    """Assert that all date values in a column are Monday-aligned."""
    bad_weekdays = df.loc[df[date_col].dt.weekday != 0, date_col].drop_duplicates()
    if not bad_weekdays.empty:
        raise ValueError(f"Found non-Monday values in {date_col}: {bad_weekdays.head().tolist()}")

def get_weeks_since(start_col: pd.Series, end_col: pd.Series) -> pd.Series:
    """Calculate the number of weeks between two dates."""
    return ((start_col - end_col).dt.days // 7).astype("int64")

def run_local_linear_rdd(df: pd.DataFrame, outcome_col: str, h: float = None, donut_weeks: list = None, label: str = "", cluster_col: str = None):
    """
    Implements a Local Linear RDD using Weighted Least Squares (WLS) with a triangular kernel.
    Specification: log(1+Y) ~ alpha + beta*Pre + gamma1*Dist + gamma2*Dist*Pre
    """
    sub = df.copy()
    if donut_weeks:
        sub = sub[~sub["dist_to_cutoff"].isin(donut_weeks)]
    
    sub = sub[(sub["dist_to_cutoff"] >= -h) & (sub["dist_to_cutoff"] <= h)].copy()
    sub = sub.dropna(subset=[outcome_col])
    
    if len(sub) < 10:
        return {"Label": label, "Outcome": outcome_col, "Estimate": np.nan, "Std.Err": np.nan, "P-value": np.nan, "N": len(sub), "BW": h, "Method": "WLS"}

    if outcome_col.startswith("cum") or "downloads" in outcome_col:
        sub["y"] = np.log1p(sub[outcome_col])
    else:
        sub["y"] = sub[outcome_col]

    sub["x"] = sub["dist_to_cutoff"]
    sub["treated"] = (sub["x"] >= 0).astype(int)
    sub["x_treated"] = sub["x"] * sub["treated"]
    sub["weight"] = 1 - (np.abs(sub["x"]) / h)
    sub = sub[sub["weight"] > 0]
    
    try:
        cov_type = 'HC1'
        fit_kwargs = {'cov_type': cov_type}
        if cluster_col and cluster_col in sub.columns:
            sub['cluster_idx'] = sub[cluster_col].astype('category').cat.codes
            fit_kwargs['cov_type'] = 'cluster'
            fit_kwargs['cov_kwds'] = {'groups': sub['cluster_idx']}

        model = smf.wls("y ~ treated + x + x_treated", data=sub, weights=sub["weight"]).fit(**fit_kwargs)
        return {"Label": label, "Outcome": outcome_col, "Estimate": model.params["treated"], "Std.Err": model.bse["treated"], "P-value": model.pvalues["treated"], "N": int(model.nobs), "BW": h, "Method": f"WLS ({fit_kwargs['cov_type']})"}
    except Exception as e:
        return {"Label": label, "Outcome": outcome_col, "Estimate": np.nan, "Std.Err": np.nan, "P-value": np.nan, "N": 0, "BW": h, "Method": "WLS"}

## Research Design: Parameter Selection & Rationale

This section details the technical and econometric justifications for the key parameters used in this analysis, as mandated by the project's measurement discipline.

### 1. Outcome Horizons (12, 26, 52 Weeks)
- **Value:** `[12, 26, 52]`
- **Rationale:** 52 weeks captures "long-run" diffusion, distinguishing durable adoption from initial launch spikes. 12 and 26-week horizons allow for the study of **diffusion dynamics**—specifically whether effects are temporary delays or persistent exclusion.

### 2. Bandwidth (h = 26 Weeks)
- **Value:** `26` weeks
- **Rationale:** Balances the **bias-variance tradeoff**. 26 weeks (6 months) is a wide enough window to maintain statistical power while remaining narrow enough to avoid major seasonal confounders that might bias the release-date effect.

### 3. Donut Hole Specification ([-2, -1, 0, 1, 2])
- **Value:** `[-2, -1, 0, 1, 2]` weeks
- **Rationale:** Accounts for **measurement error** in the release-week proxy and **conceptual ambiguity** near the cutoff. Excluding libraries released within ±2 weeks ensures we compare "definitely treated" (pre-cutoff) vs. "definitely control" (post-cutoff) libraries.

### 4. Minimum Download Thresholds ([0, 10, 100])
- **Value:** `[0, 10, 100]`
- **Rationale:** Filters out "toy" libraries and abandoned projects. Reporting results across these thresholds ensures the findings are robust and not driven solely by successful outliers or high-frequency "spam" packages.

### 5. Running Variable Alignment (Monday)
- **Value:** `W-MON` (Monday alignment)
- **Rationale:** Software development follows a strict weekly cycle. Aligning all data to Mondays ensures every "week" contains a consistent mix of workdays and weekends, removing day-of-week bias from download counts.

### 6. Kernel Choice (Triangular)
- **Value:** `Triangular`
- **Rationale:** Prioritizes libraries released **closest to the cutoff**. This aligns with the core RDD identifying assumption that the potential for omitted variable bias is minimized at the threshold.

### 7. Transformation (log(1 + y))
- **Value:** `log(1 + y)`
- **Rationale:** Addresses the **Power Law distribution** of library downloads. It prevents extreme outliers from dominating the estimates and allows the coefficients to be interpreted as approximate **percentage changes** in diffusion.

## 2.1 Identification Assumption Checklist

To interpret the RDD coefficients causally, we explicitly state and validate the following identifying assumptions:

1. **Continuity:** We assume that in the absence of the LLM cutoff, the potential adoption of libraries would be a smooth function of their release date. Any abrupt jump at the cutoff is attributed to the treatment (LLM inclusion).
2. **No Manipulation:** Developers cannot perfectly predict or manipulate their release date to fall exactly before a secret, retrospective training cutoff. The 'Donut' design further mitigates concerns about strategic timing near the boundary.
3. **Exogeneity:** No other major shock to the Python ecosystem occurred on September 27, 2021 (e.g., a major PyPI infrastructure change or a competing tool launch), that would differentially affect libraries released just before vs. just after the cutoff.
4. **Monotonicity (Fuzzy RDD Context):** Being released before the cutoff increases the probability of inclusion in the LLM's training set, and we assume no library was *less* likely to be included because it was released earlier.

## 3. Data Construction: PyPI and GitHub
We construct a library-level dataset. The **release week** is proxied by the first week with positive downloads on PyPI. We then aggregate downloads and GitHub imports over the first 52 weeks following release.

In [ ]:
# 3.1 Build PyPI Base
pypi_path = RAW_DIR / "pypi_downloads.parquet"
df_pypi = pd.read_parquet(pypi_path, columns=["project", "week_start", "downloads"])
df_pypi["package"] = normalize_name(df_pypi["project"])
df_pypi["week_start"] = pd.to_datetime(df_pypi["week_start"])
df_pypi["downloads"] = pd.to_numeric(df_pypi["downloads"], errors="coerce").fillna(0).astype("int64")

# Release Proxy
valid_downloads = df_pypi.loc[df_pypi["downloads"] > 0, ["package", "week_start"]].copy()
release_dates = valid_downloads.groupby("package", as_index=False)["week_start"].min().rename(columns={"week_start": "release_week"})

# Aggregate PyPI Horizons
df_pypi = df_pypi.merge(release_dates, on="package", how="inner")
df_pypi["weeks_since_release"] = get_weeks_since(df_pypi["week_start"], df_pypi["release_week"])
df_pypi_post = df_pypi.loc[df_pypi["weeks_since_release"] >= 0].copy()

pypi_base = release_dates.copy()
for h in [12, 26, 52]:
    agg = df_pypi_post.loc[df_pypi_post["weeks_since_release"] < h].groupby("package")["downloads"].sum().reset_index()
    pypi_base = pypi_base.merge(agg.rename(columns={"downloads": f"cum_downloads_{h}wk"}), on="package", how="left").fillna(0)

pypi_base["total_downloads_52wk"] = pypi_base["cum_downloads_52wk"]

# 3.2 Aggregate GitHub
gh_path = RAW_DIR / "github_library_week_panel.csv"
df_gh = pd.read_csv(gh_path)
df_gh["package"] = normalize_name(df_gh["library"])
df_gh["week_start"] = pd.to_datetime(df_gh["week_start"])
df_gh["import_count"] = pd.to_numeric(df_gh["import_count"], errors="coerce").fillna(0)

df_gh = df_gh.merge(release_dates, on="package", how="inner")
df_gh["weeks_since_release"] = get_weeks_since(df_gh["week_start"], df_gh["release_week"])
df_gh_post = df_gh.loc[df_gh["weeks_since_release"] >= 0].copy()

gh_agg = pd.DataFrame({"package": pypi_base["package"].unique()})
for h in [12, 26, 52]:
    agg = df_gh_post.loc[df_gh_post["weeks_since_release"] < h].groupby("package")["import_count"].sum().reset_index()
    gh_agg = gh_agg.merge(agg.rename(columns={"import_count": f"cum_imports_{h}wk"}), on="package", how="left")

# AI Score (if available)
if "ai_score_mean" in df_gh.columns:
    ai_agg = df_gh_post.loc[df_gh_post["weeks_since_release"] < 52].groupby("package")["ai_score_mean"].mean().reset_index()
    gh_agg = gh_agg.merge(ai_agg.rename(columns={"ai_score_mean": "avg_ai_score_52wk"}), on="package", how="left")

## 4. Merging and Sample Selection
We merge PyPI and GitHub data for the primary cutoff (Sept 2021). We apply bandwidth and censoring restrictions: libraries must be released within ±52 weeks of the cutoff and must have 52 weeks of available data.

In [ ]:
def prep_analysis_data(date, pypi_base, gh_agg):
    df = pypi_base.merge(gh_agg, on="package", how="left", indicator="gh_merge")
    df["matched_to_github"] = (df["gh_merge"] == "both").astype(int)
    df["dist_to_cutoff"] = get_weeks_since(df["release_week"], date)
    df["is_pre_cutoff"] = (df["dist_to_cutoff"], date)
    df["is_pre_cutoff"] = (df["dist_to_cutoff"] < 0).astype(int)
    df["in_donut"] = df["dist_to_cutoff"].isin(DONUT_WEEKS).astype(int)
    
    # Time window restriction
    df = df.loc[(df["dist_to_cutoff"].abs() <= WINDOW_WEEKS)].copy()
    
    # Data availability (Censoring check)
    pypi_max = df_pypi["week_start"].max()
    df["last_needed"] = df["release_week"] + pd.to_timedelta(51 * 7, unit="D")
    df = df.loc[df["last_needed"] <= pypi_max].copy()
    
    return df

df_main = prep_analysis_data(MAIN_CUTOFF_DATE, pypi_base, gh_agg)
print(f"Sample Size (Main 2021): {len(df_main):,}")

### 4.1 Sample Attrition Log
To ensure transparency, we track the reduction in sample size from the raw data to the final analysis set.

In [ ]:
attrition = {
    "Total PyPI Packages": len(pypi_base),
    "Outside BW Window (+/- 52 weeks)": len(pypi_base) - len(pypi_base[pypi_base['release_week'].between(MAIN_CUTOFF_DATE - pd.Timedelta(weeks=WINDOW_WEEKS), MAIN_CUTOFF_DATE + pd.Timedelta(weeks=WINDOW_WEEKS))]),
    "Dropped (Censored: < 52 weeks of data)": len(df_main[df_main['last_needed'] > df_pypi['week_start'].max()]),
    "Final Analysis Sample (N)": len(df_main),
    "GitHub Match Rate (%)": f"{(df_main['matched_to_github'].mean() * 100):.1f}%"
}

for step, count in attrition.items():
    print(f"{step:<40}: {count}")

## 5. Diagnostics (Identification Checks)
We verify the continuity of the running variable (McCrary-style density test) and the distribution of outcomes.

In [ ]:
# 5.1 Density Plot
counts = df_main.groupby("dist_to_cutoff").size()
plt.bar(counts.index, counts.values, color="steelblue", alpha=0.7)
plt.axvline(0, color="red", linestyle="--", label="Cutoff")
plt.title("Density of Python Library Releases (Running Variable)")
plt.xlabel("Weeks from Cutoff")
plt.ylabel("Count")
plt.legend()
plt.show()

# 5.2 Binned Scatterplot
df_main["log_downloads"] = np.log1p(df_main["total_downloads_52wk"])
binscatter = df_main.groupby("dist_to_cutoff")["log_downloads"].mean().reset_index()
sns.scatterplot(data=binscatter, x="dist_to_cutoff", y="log_downloads", color="darkblue")
plt.axvline(0, color="red", linestyle="--")
plt.title("Binned Scatter: Log Downloads by Release Week")
plt.show()

## 6. Main RDD Results
We estimate the treatment effect using local linear regressions with different bandwidths and sample thresholds (e.g., libraries with at least 10 downloads).

In [ ]:
results = []
for thresh in [0, 10, 100]:
    sub = df_main[df_main["total_downloads_52wk"] >= thresh]
    res = run_local_linear_rdd(sub, "total_downloads_52wk", h=26, donut_weeks=DONUT_WEEKS, label=f"Min {thresh} DL", cluster_col="dist_to_cutoff")
    results.append(res)

res_df = pd.DataFrame(results)
print("Main RDD Results (Table 2/3 equivalent):")
print(res_df.round(4))

# 6.1 Interpretation of Magnitudes
coef = res_df.loc[res_df['Label'] == 'Min 10 DL', 'Estimate'].iloc[0]
percentage = (np.exp(coef) - 1) * 100
print(f"\nInterpretation (Min 10 DL): A coefficient of {coef:.3f} implies that libraries released just AFTER the cutoff ")
print(f"experienced a {abs(percentage):.1f}% {'reduction' if percentage < 0 else 'increase'} in 52-week downloads ")
print("relative to those released just before, assuming identical potential outcomes.")

## 10. Advanced Analysis: Catch-up Dynamics and Distributional Effects

Following faculty feedback, we extend the analysis to test for **diffusion catch-up** and **distributional robustness** using Quantile RDD. These tests ensure the results are not driven solely by outliers and investigate if the LLM cutoff creates permanent exclusion or merely temporary adoption delays.

In [ ]:
# Load finalized results from the automated pipeline
final_res_path = RESULTS_DIR / "estimation_results_final.csv"
if final_res_path.exists():
    final_df = pd.read_csv(final_res_path)
    print("Final Estimation Summary (Expanded):")
    display(final_df[["Label", "Outcome", "Estimate", "Std.Err", "P-value", "Method"]].round(4))
else:
    print("Pipeline results not found. Please run scripts/05_estimation.py first.")

### 10.1 Catch-up Dynamics (Horizon Comparison)
We compare the RDD estimate at 12, 26, and 52 weeks. A move from a negative coefficient at 12 weeks to a positive one at 52 weeks supports the **Catch-up Hypothesis** (temporary delay).

In [ ]:
horizon_plot_path = FIGURES_DIR / "rdd_horizon_coefficients.png"
if horizon_plot_path.exists():
    display(Image(filename=str(horizon_plot_path)))
    print("Figure: RDD estimates across horizons. Note the recovery from 12w to 52w.")

binscatter_horizon_path = FIGURES_DIR / "rdd_horizon_binscatters.png"
if binscatter_horizon_path.exists():
    display(Image(filename=str(binscatter_horizon_path)))
    print("Figure: Faceted binscatters showing the evolution of the discontinuity.")

### 10.2 Distributional Robustness (Quantile RDD)
While log-linear models estimate the mean, **Quantile RDD** (e.g., Median) is robust to extreme outliers. We test the effect at the 25th, 50th (Median), 75th, and 90th percentiles of the adoption distribution.

In [ ]:
quantile_plot_path = FIGURES_DIR / "rdd_quantile_coefficients.png"
if quantile_plot_path.exists():
    display(Image(filename=str(quantile_plot_path)))
    print("Figure: RDD estimates by adoption percentile. The positive effect is robust across the distribution.")

## 11. Historical Context: Stacked RDD and the "Relative Suppression" Hypothesis

To determine if the 2021 discontinuity is truly unique, we pool all pre-LLM placebo cutoffs (2018-2020) and compare them to the 2021 results. This analysis introduces the **Relative Suppression** framework: the LLM effect may not be a simple dip, but the suppression of the natural adoption momentum that new libraries typically enjoy in the fourth quarter.

In [ ]:
# Load stacked results
stacked_res_path = RESULTS_DIR / "stacked_rdd_results.csv"
if stacked_res_path.exists():
    stacked_df = pd.read_csv(stacked_res_path)
    print("Stacked RDD Results (Historical Average vs. 2021):")
    display(stacked_df[["Label", "Estimate", "Std.Err", "P-value", "N"]].round(4))
else:
    print("Stacked results not found. Please run scripts/13_stacked_rdd.py first.")

### 11.1 The "Missing Boost"
Historically, libraries released in late September enjoy a significant "seasonal jump" (+1,694 downloads at the median). In 2021, this jump was **suppressed by 90%** (+169 downloads). This suggests that the LLM training cutoff significantly hindered the natural discovery of new tools during a high-growth period.

In [ ]:
stacked_plot_path = FIGURES_DIR / "stacked_rdd_comparison.png"
if stacked_plot_path.exists():
    display(Image(filename=str(stacked_plot_path)))
    print("Figure: Stacked comparison. Note the drastic reduction in the 'post-cutoff' bonus in 2021.")

## 12. Conclusion and Final Interpretation

The complete empirical evidence supports a nuanced version of the LLM diffusion hypothesis:
1. **Relative Suppression:** The primary effect of the LLM knowledge cutoff was to suppress the natural seasonal adoption boost that new Python libraries typically receive in late September. The 2021 jump was an order of magnitude smaller than the 2018-2020 historical average.
2. **Temporary vs. Permanent:** The horizon analysis shows an initial adoption penalty (-5% at 12 weeks) that fully recovers and turns positive (+3.5% at 52 weeks). This suggests that while LLM exclusion hinders immediate discovery, the market eventually 'catches up' through other channels.
3. **Distributional Consistency:** The findings are robust across the adoption spectrum (from Q25 to Q90), confirming that the 'cutoff effect' is a systemic feature of the software ecosystem in the age of generative AI.